# 📚 SQL Ch.3 — JOIN
> BigQuery SQL Reference Guide, Chapter 3: INNER · LEFT · RIGHT · FULL OUTER · SELF · CROSS JOIN  
> BigQuery SQL 완전 참조 가이드 3장: INNER · LEFT · RIGHT · FULL OUTER · SELF · CROSS JOIN

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배우고 싶은 것:
- [x] Combine two tables with `INNER`/`LEFT`/`RIGHT`/`FULL OUTER JOIN` and predict how many rows each produces  
`INNER`/`LEFT`/`RIGHT`/`FULL OUTER JOIN`로 두 테이블을 결합하고 각각 몇 행이 나올지 예측한다
- [x] Chain 3+ tables together and explain why row counts can multiply along the way  
3개 이상의 테이블을 연결하고 그 과정에서 행 수가 왜 늘어날 수 있는지 설명한다
- [x] Run a post-JOIN quality check to catch unmatched rows before trusting the result  
JOIN 직후 품질 점검을 실행해 매칭 실패 행을 결과를 신뢰하기 전에 잡아낸다

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**EN:** `JOIN` combines rows from two (or more) tables based on a matching condition, usually "this ID in table A equals that ID in table B." Different `JOIN` types answer different questions about what to do when a match is *missing*: keep only matches (`INNER`), keep everything from one side (`LEFT`/`RIGHT`), keep everything from both sides (`FULL OUTER`), or skip matching entirely and pair every row with every row (`CROSS`).

**KR:** `JOIN`은 매칭 조건 — 보통 "A 테이블의 이 ID가 B 테이블의 저 ID와 같다" — 을 기준으로 두 개(또는 그 이상) 테이블의 행을 결합합니다. JOIN 종류마다 매칭이 *없을 때* 어떻게 할지에 대한 답이 다릅니다: 매칭된 것만 남기거나(`INNER`), 한쪽은 전부 남기거나(`LEFT`/`RIGHT`), 양쪽 모두 전부 남기거나(`FULL OUTER`), 아예 매칭을 따지지 않고 모든 행끼리 짝을 짓거나(`CROSS`)입니다.

## Why do we use it?
*(When is it useful?)*

**EN:** Real business data is almost never in one table — customer info lives in `customers`, orders in `orders`, line items in `order_items`. `JOIN` is how you reassemble the full picture ("who bought what, for how much") without duplicating customer details into every single order row.

**KR:** 실무 데이터는 거의 절대 한 테이블에 다 들어있지 않습니다 — 고객 정보는 `customers`에, 주문은 `orders`에, 품목은 `order_items`에 나뉘어 있습니다. `JOIN`은 고객 정보를 모든 주문 행마다 중복시키지 않고도 전체 그림("누가 무엇을 얼마에 샀는지")을 다시 조립하는 방법입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**EN:** Nearly every non-trivial BA query involves a `JOIN`. "Revenue by customer tier" needs `orders` joined to `customers`. "Which employees have no direct reports" needs a `SELF JOIN`. `LEFT JOIN` in particular is the standard tool for finding *gaps* — customers who never ordered, products that never sold, employees with no manager on file.

**KR:** 사소하지 않은 BA 쿼리는 거의 다 `JOIN`이 들어갑니다. "고객 등급별 매출"은 `orders`와 `customers`를 조인해야 하고, "직속 부하직원이 없는 직원"은 `SELF JOIN`이 필요합니다. 특히 `LEFT JOIN`은 *빈틈*을 찾는 표준 도구입니다 — 한 번도 주문하지 않은 고객, 한 번도 팔리지 않은 상품, 관리자 정보가 없는 직원 같은 것들입니다.

**Comparison / 비교표:**

| Task / 작업 | SQL | Pandas | Excel |
|---|---|---|---|
| Only matches / 매칭된 것만 | `INNER JOIN` | `pd.merge(how="inner")` | VLOOKUP (exact match) |
| Keep left, NULL if unmatched / 왼쪽 유지 | `LEFT JOIN` | `pd.merge(how="left")` | VLOOKUP + IFERROR |
| Keep right, NULL if unmatched / 오른쪽 유지 | `RIGHT JOIN` | `pd.merge(how="right")` | (swap tables) |
| Keep everything / 전부 유지 | `FULL OUTER JOIN` | `pd.merge(how="outer")` | (no direct equivalent) |
| Every combination / 모든 조합 | `CROSS JOIN` | `pd.merge(how="cross")` | (no direct equivalent) |

---
# 📝 Syntax

## Basic Syntax
`INNER JOIN` — the strictest kind: a row survives only if it matches on *both* sides.
`INNER JOIN` — 가장 엄격한 종류: *양쪽 모두* 매칭되는 행만 살아남습니다.

In [1]:
# --- Environment setup / 환경 설정 ---
# We use DuckDB: a free, in-memory SQL engine that understands BigQuery-style syntax
# almost 1:1 (window functions, QUALIFY, ROLLUP, STRING_AGG, etc.), and can query
# pandas DataFrames directly by name -- no separate "load data" step needed.
# DuckDB는 무료 인메모리 SQL 엔진으로, BigQuery 문법(윈도우 함수, QUALIFY, ROLLUP,
# STRING_AGG 등)을 거의 그대로 이해하고, pandas DataFrame을 이름으로 바로 조회할 수
# 있습니다. 별도의 "데이터 로드" 단계가 필요 없습니다.
import duckdb
import pandas as pd
from IPython.display import display

def run(sql: str) -> pd.DataFrame:
    """Execute a SQL string against DuckDB and return the result as a DataFrame.
    SQL 문자열을 DuckDB에서 실행하고 결과를 DataFrame으로 반환합니다."""
    return duckdb.sql(sql).df()

customers = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C05"],   # note: no C04 here / C04는 여기 없음
    "name":        ["김민수", "이영희", "박준호", "최서연"],
    "region":      ["서울", "부산", "서울", "인천"],
})
orders = pd.DataFrame({
    "order_id":    [1001, 1002, 1003, 1004, 1005],
    "customer_id": ["C01", "C02", "C01", "C04", "C02"],  # note: C04 has no matching customer / C04는 customers에 없음
    "amount":      [45000, 32000, 61000, 28000, 95000],
})

sql = """
SELECT o.order_id, o.customer_id, c.name, c.region, o.amount
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id
"""
display(run(sql))
# order 1004 (C04, no matching customer) and customers C03/C05 (no orders) are BOTH dropped -> 4 rows
# 주문 1004(C04, 매칭 고객 없음)와 고객 C03·C05(주문 없음) 모두 제외됨 -> 4행


,order_id,customer_id,name,region,amount
0,1001,C01,김민수,서울,45000
1,1002,C02,이영희,부산,32000
2,1003,C01,김민수,서울,61000
3,1005,C02,이영희,부산,95000


## Common Variations

In [2]:
# INNER is the default -- writing just "JOIN" means INNER JOIN. Explicit is still better style.
# INNER는 기본값 -- "JOIN"만 써도 INNER JOIN. 그래도 명시하는 게 스타일상 더 좋습니다.
display(run("SELECT o.order_id, c.name FROM orders o JOIN customers c ON o.customer_id = c.customer_id"))

# If the join key has the same name on both sides, USING() is a shorter alternative to ON
# 조인 키 이름이 양쪽에서 같다면, ON 대신 USING()으로 더 짧게 쓸 수 있음
display(run("SELECT o.order_id, c.name FROM orders o JOIN customers c USING(customer_id)"))


,order_id,name
0,1001,김민수
1,1002,이영희
2,1003,김민수
3,1005,이영희


,order_id,name
0,1001,김민수
1,1002,이영희
2,1003,김민수
3,1005,이영희


---
# 🧪 Small Examples

## Example 1 — LEFT JOIN: All Left Rows, NULL on No Match / 왼쪽 기준, 매칭 없으면 NULL
**EN:** `LEFT JOIN` keeps **every** row from the left (first-listed) table, no matter what — if there's no match on the right, the right-side columns simply come back `NULL`. This makes `LEFT JOIN` the standard tool for "show me everything, and tell me what's missing."  
**KR:** `LEFT JOIN`은 왼쪽(먼저 쓴) 테이블의 행을 **무조건 전부** 유지합니다 — 오른쪽에 매칭이 없으면 오른쪽 열은 그냥 `NULL`로 채워집니다. 그래서 `LEFT JOIN`은 "전부 보여주되, 뭐가 빠졌는지도 알려줘"의 표준 도구입니다.

In [3]:
sql = """
SELECT o.order_id, o.customer_id, c.name, c.region, o.amount
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_id
"""
display(run(sql))
# All 5 orders survive. Order 1004 (customer C04) has NULL name/region -- no matching customer.
# Customers C03/C05 (no orders) never appear -- orders is the LEFT table here, so it sets the row count.
# 5개 주문이 모두 살아남음. 1004(C04)는 매칭되는 고객이 없어 name/region이 NULL.
# 고객 C03·C05(주문 없음)는 애초에 나타나지 않음 -- orders가 왼쪽 테이블이라 행 수를 이게 결정.


,order_id,customer_id,name,region,amount
0,1001,C01,김민수,서울,45000
1,1002,C02,이영희,부산,32000
2,1003,C01,김민수,서울,61000
3,1004,C04,NaN,NaN,28000
4,1005,C02,이영희,부산,95000


## Example 2 — RIGHT JOIN: All Right Rows, the Mirror of LEFT / 오른쪽 기준 (LEFT JOIN의 정반대)
**EN:** `RIGHT JOIN` is `LEFT JOIN` with the tables' roles swapped — it keeps every row from the right (second-listed) table. `A RIGHT JOIN B` always gives the exact same result as `B LEFT JOIN A`, which is why many teams standardize on `LEFT JOIN` alone (just reorder the tables) for readability.  
**KR:** `RIGHT JOIN`은 테이블의 역할이 뒤바뀐 `LEFT JOIN`입니다 — 오른쪽(두 번째로 쓴) 테이블의 행을 전부 유지합니다. `A RIGHT JOIN B`는 항상 `B LEFT JOIN A`와 정확히 같은 결과이므로, 많은 팀이 가독성을 위해 `LEFT JOIN` 하나로(테이블 순서만 바꿔서) 통일합니다.

In [4]:
sql = """
SELECT o.order_id, c.customer_id, c.name, c.region, o.amount
FROM orders o
RIGHT JOIN customers c ON o.customer_id = c.customer_id
ORDER BY c.customer_id, o.order_id
"""
display(run(sql))
# All 4 customers survive. C03/C05 (no orders) have NULL order_id/amount.
# Order 1004 (customer C04) is dropped -- customers is now the side that sets the row count.
# 4명 고객이 모두 살아남음. C03·C05(주문 없음)는 order_id/amount가 NULL.
# 1004(C04)는 제외됨 -- 이번엔 customers가 행 수를 결정.

# LEFT vs RIGHT, side by side / LEFT vs RIGHT 직접 비교:
#   LEFT (orders as base)     -> 1004 (C04's order) stays,   C03/C05 (no-order customers) drop
#   RIGHT (customers as base) -> 1004 (C04's order) drops,   C03/C05 (no-order customers) stay


,order_id,customer_id,name,region,amount
0,1001,C01,김민수,서울,45000
1,1003,C01,김민수,서울,61000
2,1002,C02,이영희,부산,32000
3,1005,C02,이영희,부산,95000
4,<NA>,C03,박준호,서울,<NA>
5,<NA>,C05,최서연,인천,<NA>


## Example 3 — FULL OUTER JOIN: Everything From Both Sides / 양쪽 전부, 매칭 없으면 양쪽 다 NULL 가능
**EN:** `FULL OUTER JOIN` keeps every row from *both* tables — matched rows join normally, and unmatched rows from either side appear with `NULL` on the other side. It's the tool for finding "orphans" in *both* directions at once (BigQuery/PostgreSQL/Snowflake all support it; MySQL doesn't — it must be simulated with `LEFT JOIN ... UNION ... RIGHT JOIN`).  
**KR:** `FULL OUTER JOIN`은 *양쪽* 테이블의 모든 행을 유지합니다 — 매칭된 행은 정상적으로 결합되고, 어느 한쪽에서 매칭 안 된 행은 반대쪽이 `NULL`로 채워져 나타납니다. 양방향 "고아 데이터"를 한 번에 찾는 도구입니다(BigQuery/PostgreSQL/Snowflake는 모두 지원하지만 MySQL은 지원하지 않아 `LEFT JOIN ... UNION ... RIGHT JOIN`으로 흉내내야 합니다).

In [5]:
sql = """
SELECT
    o.order_id,
    o.customer_id AS order_side_id,
    c.customer_id AS customer_side_id,
    c.name,
    o.amount
FROM orders o
FULL OUTER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY COALESCE(o.customer_id, c.customer_id), o.order_id
"""
display(run(sql))
# order_side_id=C04 / customer_side_id=NULL -> an order with no matching customer (a "bad" order)
# customer_side_id=C03/C05 / order_side_id=NULL -> customers who never ordered
# order_side_id=C04 / customer_side_id=NULL -> 매칭 고객이 없는 주문 ("잘못된" 주문)
# customer_side_id=C03/C05 / order_side_id=NULL -> 한 번도 주문 안 한 고객
print()
print("4-way row-count comparison / 4가지 JOIN 행 수 한눈에 비교:")
comparison = pd.DataFrame({
    "JOIN type / 종류": ["INNER", "LEFT (orders-based)", "RIGHT (customers-based)", "FULL OUTER"],
    "Result rows / 결과 행 수": [4, 5, 6, 7],
    "Excludes / 제외되는 것": ["C04(order-only), C03/C05(customer-only)", "C03/C05(customer-only)", "C04(order-only)", "nothing / 없음"],
})
display(comparison)


,order_id,order_side_id,customer_side_id,name,amount
0,1001,C01,C01,김민수,45000
1,1003,C01,C01,김민수,61000
2,1002,C02,C02,이영희,32000
3,1005,C02,C02,이영희,95000
4,<NA>,NaN,C03,박준호,<NA>
5,1004,C04,NaN,NaN,28000
6,<NA>,NaN,C05,최서연,<NA>



4-way row-count comparison / 4가지 JOIN 행 수 한눈에 비교:


,JOIN type / 종류,Result rows / 결과 행 수,Excludes / 제외되는 것
0,INNER,4,"C04(order-only), C03/C05(customer-only)"
1,LEFT (orders-based),5,C03/C05(customer-only)
2,RIGHT (customers-based),6,C04(order-only)
3,FULL OUTER,7,nothing / 없음


## Example 4 — Multi-Table JOIN: Chaining 3+ Tables / 다중 테이블 JOIN
**EN:** Join more than two tables by chaining `JOIN` clauses one after another — each `ON` connects to whichever table already appeared just above it. **Watch the row count**: joining through a one-to-many relationship (one order → many line items) expands the row count, exactly like pandas' `merge()` does on a 1:N key.  
**KR:** 두 개보다 많은 테이블은 `JOIN`을 체인처럼 이어 붙여서 연결합니다 — 각 `ON`은 바로 위에 나온 테이블과 연결됩니다. **행 수를 주의 깊게 보세요**: 1대다 관계(주문 하나 → 품목 여러 개)를 거치면 행 수가 늘어나는데, pandas의 `merge()`가 1:N 키에서 하는 동작과 똑같습니다.

In [6]:
customers2 = pd.DataFrame({"customer_id": ["C01", "C02"], "name": ["김민수", "이영희"]})
orders2 = pd.DataFrame({"order_id": [1001, 1002], "customer_id": ["C01", "C02"]})
order_items = pd.DataFrame({
    "order_id":     [1001, 1001, 1002],
    "product_name": ["노트북", "마우스", "키보드"],
    "quantity":     [1, 2, 1],
    "unit_price":   [1200000, 25000, 45000],
})

sql = """
SELECT
    c.name,
    o.order_id,
    oi.product_name,
    oi.quantity,
    oi.unit_price,
    oi.quantity * oi.unit_price AS line_total
FROM customers2 c
JOIN orders2 o      ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id    = oi.order_id
ORDER BY o.order_id
"""
display(run(sql))
# order 1001 has 2 line items, so it expands from 1 order-row into 2 result-rows -> 3 rows total, not 2
# 1001 주문은 품목이 2개라 주문 1행이 결과 2행으로 늘어남 -> 전체 3행 (2행이 아님)


,name,order_id,product_name,quantity,unit_price,line_total
0,김민수,1001,노트북,1,1200000,1200000
1,김민수,1001,마우스,2,25000,50000
2,이영희,1002,키보드,1,45000,45000


## Example 5 — SELF JOIN: Joining a Table to Itself / 같은 테이블끼리 조인 (계층 구조)
**EN:** A `SELF JOIN` joins a table to a copy of itself, using two different aliases to view it from two angles at once — perfect for hierarchies, like "each employee, next to their manager's name" (where the manager is *also* a row in the same `employees` table). Use `LEFT JOIN` here, not `INNER`, so employees with no manager (the CEO) aren't dropped.  
**KR:** `SELF JOIN`은 테이블을 그 자신의 복사본과 조인하는데, 서로 다른 별칭 두 개로 동시에 두 관점에서 바라보는 방식입니다 — 계층 구조에 딱 맞습니다, 예를 들어 "각 직원 옆에 그 매니저의 이름을 붙이기"(매니저도 같은 `employees` 테이블의 한 행). 관리자가 없는 직원(대표)이 사라지지 않도록 `INNER`가 아니라 `LEFT JOIN`을 씁니다.

In [7]:
employees = pd.DataFrame({
    "emp_id":     ["E01", "E02", "E03", "E04", "E05"],
    "name":       ["김민수", "이영희", "박준호", "최서연", "정대현"],
    "manager_id": [None, "E01", "E01", "E02", "E02"],
})

sql = """
SELECT e.name AS employee, m.name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
"""
display(run(sql))
# The same table "employees" is referenced twice -- once as "e" (employee view), once as "m" (manager view).
# 김민수 has no manager (manager_id IS NULL), so LEFT JOIN keeps that row with manager=NULL --
# an INNER JOIN would have silently dropped 김민수 entirely.
# 같은 "employees" 테이블을 두 번 참조 -- 한 번은 "e"(직원 관점), 한 번은 "m"(매니저 관점).
# 김민수는 매니저가 없어서(manager_id가 NULL) LEFT JOIN이 manager=NULL로 그 행을 유지함 --
# INNER JOIN이었다면 김민수 행이 통째로 조용히 사라졌을 것.


,employee,manager
0,이영희,김민수
1,박준호,김민수
2,최서연,이영희
3,정대현,이영희
4,김민수,NaN


## Example 6 — CROSS JOIN: Every Combination / 모든 조합 (카테시안 곱)
**EN:** `CROSS JOIN` pairs *every* row of one table with *every* row of the other, with no matching condition at all — `n × m` result rows. Used intentionally, it's great for generating a full combination grid (e.g. every size × every color, for an inventory template). Used *by accident* — usually by listing tables comma-separated with no `ON`/`WHERE` — it silently multiplies your row count and gives no error, which makes it dangerous.  
**KR:** `CROSS JOIN`은 매칭 조건 없이 한 테이블의 *모든* 행을 다른 테이블의 *모든* 행과 짝짓습니다 — 결과는 `n × m`행. 의도적으로 쓰면 전체 조합 그리드를 만들 때 유용합니다(예: 모든 사이즈 × 모든 색상, 재고 템플릿용). *실수로* 쓰면 — 보통 `ON`/`WHERE` 없이 테이블을 쉼표로 나열해서 — 오류 없이 조용히 행 수를 곱해버리므로 위험합니다.

In [8]:
sizes = pd.DataFrame({"size": ["S", "M", "L"]})
colors = pd.DataFrame({"color": ["Red", "Blue"]})

print("-- intentional: full size x color grid / 의도적 사용: 사이즈 x 색상 전체 조합 --")
display(run("SELECT s.size, c.color FROM sizes s CROSS JOIN colors c"))
# 3 sizes x 2 colors = 6 rows, every combination / 3 x 2 = 6행, 모든 조합

print("-- ⚠️ accidental: comma-separated tables with NO join condition --")
print("-- ⚠️ 실수: 조인 조건 없이 테이블을 쉼표로만 나열 --")
sql_accident = "SELECT o.order_id, c.name FROM orders o, customers c"  # no ON! / ON 없음!
row_count = len(run(sql_accident))
print(f"rows returned: {row_count}  (orders has 5 rows, customers has 4 -> 5 x 4 = {5*4}, NOT the ~4-5 you probably wanted)")
print(f"반환된 행 수: {row_count}  (orders 5행 x customers 4행 = {5*4}행 -- 의도했을 4~5행이 아님)")


-- intentional: full size x color grid / 의도적 사용: 사이즈 x 색상 전체 조합 --


,size,color
0,S,Red
1,M,Red
2,L,Red
3,S,Blue
4,M,Blue
5,L,Blue


-- ⚠️ accidental: comma-separated tables with NO join condition --
-- ⚠️ 실수: 조인 조건 없이 테이블을 쉼표로만 나열 --
rows returned: 20  (orders has 5 rows, customers has 4 -> 5 x 4 = 20, NOT the ~4-5 you probably wanted)
반환된 행 수: 20  (orders 5행 x customers 4행 = 20행 -- 의도했을 4~5행이 아님)


## Example 7 — JOIN Quality Checks / JOIN 후 품질 점검
**EN:** A `JOIN` can quietly "lose" or "explode" rows without ever throwing an error — so it's worth building a 3-step habit: (1) count rows in each source table, (2) `LEFT JOIN` and count total rows + `NULL` matches, (3) look directly at the unmatched rows to understand *why* they didn't match.  
**KR:** `JOIN`은 오류 하나 없이 조용히 행을 "잃거나" "폭발시킬" 수 있으므로, 3단계 습관을 들여두는 게 좋습니다: (1) 원본 테이블 각각의 행 수를 세고, (2) `LEFT JOIN`한 뒤 전체 행 수와 `NULL` 매칭 수를 세고, (3) 매칭 실패 행을 직접 확인해서 *왜* 매칭이 안 됐는지 파악합니다.

In [9]:
print("-- Step 1: source table row counts / 원본 테이블 행 수 --")
display(run("SELECT (SELECT COUNT(*) FROM orders) AS orders_count, (SELECT COUNT(*) FROM customers) AS customers_count"))

print("-- Step 2: LEFT JOIN, then count total rows + unmatched (NULL) rows --")
print("-- Step 2: LEFT JOIN 후 전체 행 수 + 매칭 실패(NULL) 행 수 --")
sql2 = """
SELECT
    COUNT(*) AS total_rows,
    COUNTIF(c.name IS NULL) AS unmatched_rows   -- BigQuery shortcut for SUM(CASE WHEN ... THEN 1 ELSE 0 END)
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
"""
display(run(sql2))

print("-- Step 3: look at the unmatched rows directly / 매칭 실패 행 직접 확인 --")
sql3 = """
SELECT o.order_id, o.customer_id
FROM orders o
LEFT JOIN customers c ON o.customer_id = c.customer_id
WHERE c.name IS NULL
"""
display(run(sql3))
# order 1004 references customer C04, which simply doesn't exist in the customers table.
# 1004 주문은 customer_id가 C04인데, customers 테이블에 C04가 아예 존재하지 않음.


-- Step 1: source table row counts / 원본 테이블 행 수 --


,orders_count,customers_count
0,5,4


-- Step 2: LEFT JOIN, then count total rows + unmatched (NULL) rows --
-- Step 2: LEFT JOIN 후 전체 행 수 + 매칭 실패(NULL) 행 수 --


,total_rows,unmatched_rows
0,5,1.0


-- Step 3: look at the unmatched rows directly / 매칭 실패 행 직접 확인 --


,order_id,customer_id
0,1004,C04


## Example 8 — Common Combinations / 자주 쓰는 조합
**EN:** **Pattern A** chains a 3-table `JOIN` straight into `GROUP BY` — the standard shape for "customer lifetime value" and similar rollup metrics. **Pattern B** uses a `SELF JOIN` + `LEFT JOIN` + `COUNT` to get a manager's direct-report count, including managers with **zero** reports (an `INNER JOIN` would silently drop them).  
**KR:** **패턴 A**는 3-테이블 `JOIN`을 곧바로 `GROUP BY`로 이어붙입니다 — "고객 생애가치(LTV)" 같은 집계 지표의 표준 형태입니다. **패턴 B**는 `SELF JOIN` + `LEFT JOIN` + `COUNT`로 매니저별 직속 부하 수를 구하는데, 부하가 **0명**인 매니저까지 포함합니다(`INNER JOIN`이었다면 조용히 빠졌을 것).

In [10]:
print("-- Pattern A: 3-table JOIN + GROUP BY -- customer total spend / 고객별 총 구매액 --")
sql_a = """
SELECT c.name, SUM(oi.quantity * oi.unit_price) AS total_spent
FROM customers2 c
JOIN orders2 o      ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id    = oi.order_id
GROUP BY c.name
ORDER BY total_spent DESC
"""
display(run(sql_a))

print("-- Pattern B: SELF JOIN + COUNT -- direct reports per manager / 매니저별 부하직원 수 --")
sql_b = """
SELECT m.name AS manager, COUNT(e.emp_id) AS report_count
FROM employees m
LEFT JOIN employees e ON e.manager_id = m.emp_id
GROUP BY m.name
ORDER BY report_count DESC
"""
display(run(sql_b))
# 박준호/최서연/정대현 all show report_count=0 -- they exist, they simply have no one reporting to them.
# 박준호·최서연·정대현 모두 report_count=0 -- 존재하긴 하지만 그냥 부하직원이 없는 것.


-- Pattern A: 3-table JOIN + GROUP BY -- customer total spend / 고객별 총 구매액 --


,name,total_spent
0,김민수,1250000.0
1,이영희,45000.0


-- Pattern B: SELF JOIN + COUNT -- direct reports per manager / 매니저별 부하직원 수 --


,manager,report_count
0,이영희,2
1,김민수,2
2,박준호,0
3,정대현,0
4,최서연,0


## Example 9 — Practice / 실습 문제
**EN:** Fill in each `________` blank below, then remove the `#` in front of the matching `display(run(...))` line to check your answer. Hints: `LEFT` `NULL` `JOIN` `GROUP`
**KR:** 아래 `________` 빈칸을 채운 뒤, 해당 `display(run(...))` 줄 앞의 `#`을 지우고 실행해서 답을 확인하세요. 힌트: `LEFT` `NULL` `JOIN` `GROUP`

In [14]:
customers_p = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04"],
    "name":        ["김민수", "이영희", "박준호", "최서연"],
    "tier":        ["Gold", "Silver", "Gold", "Bronze"],
})
orders_p = pd.DataFrame({
    "order_id":    [2001, 2002, 2003, 2004],
    "customer_id": ["C01", "C01", "C02", "C05"],   # note: C05 has no matching customer / C05는 customers에 없음
    "amount":      [50000, 30000, 20000, 15000],
})

# Q1. Names of customers who have NEVER placed an order.
# Q1. 한 번도 주문하지 않은 고객의 이름.
q1 = """
SELECT c.customer_id, c.name
FROM customers_p c
LEFT JOIN orders_p o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
"""
display(run(q1))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q2. Total order amount, only for customers who have ordered at least once.
# Q2. 주문 이력이 있는 고객만, 고객별 총 주문 금액.
q2 = """
SELECT c.name, SUM(o.amount) AS total
FROM customers_p c
INNER JOIN orders_p o ON c.customer_id = o.customer_id
GROUP BY c.name
"""
display(run(q2))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

# Q3. Count "orphan" orders -- orders whose customer_id has no matching customer.
# Q3. 매칭되는 고객이 없는 "고아 주문"의 건수.
q3 = """
SELECT COUNT(*) AS orphan_orders
FROM orders_p o
LEFT JOIN customers_p c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
"""
display(run(q3))   # <- uncomment once filled in / 빈칸을 채운 뒤 주석 해제

print("✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.")
print("✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.")


,customer_id,name
0,C03,박준호
1,C04,최서연


,name,total
0,김민수,80000.0
1,이영희,20000.0


,orphan_orders
0,1


✏️  Fill in the ________ blanks above, uncomment the display() lines, then re-run this cell.
✏️  위 ________ 빈칸을 채우고 display() 줄의 주석을 해제한 뒤 이 셀을 다시 실행하세요.


<details>
<summary>🔑 Answer / 정답 (click to expand / 클릭해서 펼치기)</summary>

```sql
-- Q1
SELECT c.customer_id, c.name
FROM customers_p c
LEFT JOIN orders_p o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL

-- Q2
SELECT c.name, SUM(o.amount) AS total
FROM customers_p c
JOIN orders_p o ON c.customer_id = o.customer_id
GROUP BY c.name

-- Q3
SELECT COUNT(*) AS orphan_orders
FROM orders_p o
LEFT JOIN customers_p c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL
```
</details>

---
# ⚠️ Common Mistakes

**Mistake 1 — Forgetting the `ON` condition (accidental CROSS JOIN)**
- EN: Listing tables comma-separated, or writing `JOIN` without an `ON`/`USING`, silently produces a cartesian product — every row paired with every row. No error is raised; the row count just quietly explodes.
- KR: 테이블을 쉼표로만 나열하거나 `ON`/`USING` 없이 `JOIN`을 쓰면 조용히 카테시안 곱이 만들어집니다 — 모든 행이 모든 행과 짝지어집니다. 오류는 나지 않고, 행 수만 조용히 폭발합니다.
- ✅ Fix / 해결법: Always write an explicit `ON` (or `USING`) condition, and check the resulting row count against what you expected.  
항상 명시적인 `ON`(또는 `USING`) 조건을 쓰고, 결과 행 수가 예상과 맞는지 확인하세요.

**Mistake 2 — Using `INNER JOIN` when you actually need `LEFT JOIN`**
- EN: `INNER JOIN` silently drops any row without a match on both sides. If the goal is "show me everyone, including who's missing something," `INNER JOIN` will quietly remove exactly the rows you were looking for.
- KR: `INNER JOIN`은 양쪽에서 매칭되지 않는 행을 조용히 제거합니다. 목표가 "전부 보여주되, 뭐가 빠졌는지도 알기"라면, `INNER JOIN`은 정작 찾고 있던 행들을 조용히 없애버립니다.
- ✅ Fix / 해결법: Default to `LEFT JOIN` whenever you need to preserve every row from a "base" table, and use `WHERE ... IS NULL` to isolate the gaps.  
"기준" 테이블의 모든 행을 보존해야 한다면 `LEFT JOIN`을 기본으로 쓰고, `WHERE ... IS NULL`로 빈틈을 골라내세요.

**Mistake 3 — Not expecting row-count expansion on a 1:N JOIN**
- EN: Joining through a one-to-many relationship (one order, many line items) multiplies rows — an aggregate like `SUM(amount)` computed *before* that join can come out wrong if you're not careful about join order.
- KR: 1대다 관계(주문 하나, 품목 여러 개)를 거쳐 조인하면 행이 늘어납니다 — 조인 순서를 주의하지 않으면 그 조인 *이전에* 계산한 `SUM(amount)` 같은 집계값이 잘못 나올 수 있습니다.
- ✅ Fix / 해결법: After any multi-table JOIN, sanity-check the row count against what each table's granularity implies (Example 7's 3-step habit).  
다중 테이블 JOIN 이후에는 각 테이블의 세분성이 암시하는 값과 결과 행 수를 대조해 확인하세요(예제 7의 3단계 습관).

**Mistake 4 — Ambiguous column names across joined tables**
- EN: When two joined tables both have a column called `name` (or `id`), writing bare `name` in `SELECT`/`WHERE` causes an "ambiguous column" error — the database doesn't know which table's `name` you mean.
- KR: 조인한 두 테이블에 모두 `name`(또는 `id`) 열이 있을 때, `SELECT`/`WHERE`에 그냥 `name`이라고만 쓰면 "모호한 열" 오류가 납니다 — 어느 테이블의 `name`을 말하는지 데이터베이스가 알 수 없습니다.
- ✅ Fix / 해결법: Always qualify shared column names with a table alias, like `c.name` vs `o.name` — this is exactly why table aliases from Chapter 1 matter so much once JOIN enters the picture.  
공유되는 열 이름은 항상 `c.name` vs `o.name`처럼 테이블 별칭으로 구분하세요 — JOIN이 등장하는 순간 1장의 테이블 별칭이 왜 그렇게 중요한지 드러납니다.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- `A RIGHT JOIN B` always equals `B LEFT JOIN A` — most teams standardize on `LEFT JOIN` alone (just reorder tables) for consistent, easier-to-scan code.  
 `A RIGHT JOIN B`는 항상 `B LEFT JOIN A`와 같습니다 — 대부분의 팀은 코드 일관성과 가독성을 위해 `LEFT JOIN`(테이블 순서만 바꿔서) 하나로 통일합니다.
- After writing any JOIN, immediately run `SELECT COUNT(*)` and sanity-check it against the source tables' row counts, *before* building anything on top of the result.  
 JOIN을 작성한 직후에는 바로 `SELECT COUNT(*)`를 실행해서 원본 테이블 행 수와 대조해 보세요 — 그 결과 위에 뭔가를 더 쌓기 *전에*.
- `LEFT JOIN` + `WHERE right_table.col IS NULL` is the single most useful JOIN pattern in BA work — memorize it as one unit, not two separate ideas.  
 `LEFT JOIN` + `WHERE right_table.col IS NULL`은 BA 업무에서 가장 유용한 JOIN 패턴입니다 — 두 개의 아이디어가 아니라 하나로 묶어서 외워두세요.
- `FULL OUTER JOIN` isn't something you reach for often, but when you need to audit data quality in *both* directions at once, nothing else does the job in one query.  
 `FULL OUTER JOIN`은 자주 쓰지는 않지만, 양쪽 방향의 데이터 품질을 한 번에 점검해야 할 때는 이것만큼 한 쿼리로 끝내주는 게 없습니다.

---
# 🔗 Related Concepts

```
SQL Learning Roadmap (this guide) / SQL 학습 로드맵 (이 가이드)
──────────────────────────────────────────────
 1. SELECT Basics
 2. Aggregation & GROUP BY
 3. JOIN                        ← ★ YOU ARE HERE / 지금 여기
 4. Subquery & CTE
 5. Conditions & NULL Handling
 6. String & Date Functions
 7. Window Functions
 8. BA-Specific Patterns
```

```
The four JOIN types, visually / 네 가지 JOIN을 그림으로
──────────────────────────────────────────────
   INNER          LEFT           RIGHT          FULL OUTER
   (A)∩(B)        (A) + ∩        ∩ + (B)        (A) + ∩ + (B)

   A   B          A   B          A   B          A   B
  ┌─┐ ┌─┐        ┌▓┐ ┌─┐        ┌─┐ ┌▓┐        ┌▓┐ ┌▓┐
  │ ╳ │   -->    │▓╳ │   -->    │ ╳▓│   -->    │▓╳▓│
  └─┘ └─┘        └▓┘ └─┘        └─┘ └▓┘        └▓┘ └▓┘
  only the       all of A       all of B       all of both
  overlap        + overlap      + overlap      A and B
```

*How is today's topic connected to other concepts?*

**EN:** JOIN happens inside `FROM`, which is the very *first* step of Chapter 2's execution order — so everything from Chapter 2 (`WHERE`, `GROUP BY`, `HAVING`) applies normally to a joined result, as if it were one big table (Example 8, Pattern A, is exactly this: `JOIN` then `GROUP BY`). Looking ahead, Chapter 4 introduces subqueries and CTEs as an *alternative* way to combine data from multiple sources — sometimes cleaner than a JOIN, sometimes built on top of one.

**KR:** JOIN은 `FROM` 안에서 일어나며, 이는 2장 실행 순서의 *가장 첫* 단계입니다 — 그래서 2장의 모든 내용(`WHERE`, `GROUP BY`, `HAVING`)이 조인된 결과에도 마치 하나의 큰 테이블인 것처럼 그대로 적용됩니다(예제 8의 패턴 A가 정확히 이 경우: `JOIN` 다음 `GROUP BY`). 앞으로 배울 4장은 여러 데이터 소스를 결합하는 *또 다른* 방법인 서브쿼리와 CTE를 소개하는데, JOIN보다 더 깔끔할 때도 있고 JOIN 위에 쌓아 올릴 때도 있습니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오:**
**EN:** Customer Success asks: *"Which of our Gold-tier customers have never actually placed an order? We want to reach out to them personally."* This needs `LEFT JOIN` + `WHERE ... IS NULL`, filtered down to one tier.
**KR:** 고객 성공팀이 묻습니다: *"우리 Gold 등급 고객 중에 실제로는 한 번도 주문 안 한 사람이 누구야? 개인적으로 연락해보고 싶어."* `LEFT JOIN` + `WHERE ... IS NULL`을 쓰고, 등급 하나로 필터링하면 됩니다.

**To-do / 할 일:**
- [x] `LEFT JOIN` customers to orders, keeping every customer  
고객을 기준으로 주문에 `LEFT JOIN`해서 모든 고객을 유지한다
- [x] Keep only rows where no matching order exists  
매칭되는 주문이 없는 행만 남긴다
- [x] Filter further to `tier = 'Gold'`  
`tier = 'Gold'` 조건으로 한 번 더 거른다

In [12]:
customers_biz = pd.DataFrame({
    "customer_id": ["C01", "C02", "C03", "C04", "C05"],
    "name":        ["김민수", "이영희", "박준호", "최서연", "정대현"],
    "tier":        ["Gold", "Gold", "Silver", "Gold", "Bronze"],
})
orders_biz = pd.DataFrame({
    "order_id":    [3001, 3002, 3003],
    "customer_id": ["C01", "C03", "C05"],
    "amount":      [80000, 40000, 12000],
})

sql = """
SELECT c.customer_id, c.name, c.tier
FROM customers_biz c
LEFT JOIN orders_biz o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
  AND c.tier = 'Gold'
"""
display(run(sql))


,customer_id,name,tier
0,C04,최서연,Gold
1,C02,이영희,Gold


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**EN:** `JOIN` combines rows from multiple tables using a matching condition, and the four core types differ only in what happens to *unmatched* rows: `INNER` drops them from both sides, `LEFT`/`RIGHT` keeps everything from one named side, and `FULL OUTER` keeps everything from both. Chaining 3+ tables can expand row counts through one-to-many relationships, exactly like pandas' `merge()` does. `SELF JOIN` reuses one table under two aliases to model hierarchies, while `CROSS JOIN` deliberately (or accidentally, if you forget `ON`) pairs every row with every row. After any JOIN, a quick 3-step quality check — count sources, count total + `NULL` matches, inspect the unmatched rows directly — catches silent row loss or explosion before it reaches a report.

**KR:** `JOIN`은 매칭 조건을 사용해 여러 테이블의 행을 결합하며, 네 가지 핵심 종류는 *매칭 안 된* 행을 어떻게 처리하는지만 다릅니다: `INNER`는 양쪽에서 제거하고, `LEFT`/`RIGHT`는 지정한 한쪽은 전부 유지하며, `FULL OUTER`는 양쪽 모두 전부 유지합니다. 3개 이상의 테이블을 이어 붙이면 1대다 관계를 거치면서 행 수가 늘어날 수 있는데, pandas의 `merge()`와 똑같은 원리입니다. `SELF JOIN`은 하나의 테이블을 별칭 두 개로 재사용해서 계층 구조를 표현하고, `CROSS JOIN`은 의도적으로(또는 `ON`을 빠뜨려서 실수로) 모든 행을 모든 행과 짝짓습니다. 어떤 JOIN 뒤든 원본 개수 세기 → 전체+NULL 매칭 개수 세기 → 매칭 실패 행 직접 확인이라는 3단계 품질 점검이 리포트에 반영되기 전 조용한 행 손실이나 폭발을 잡아줍니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence. / 오늘 배운 내용을 한 문장으로.

> **EN:** Every JOIN type answers the exact same underlying question — "what should happen to a row that doesn't find a match?" — and once you can answer that for `INNER`/`LEFT`/`RIGHT`/`FULL OUTER` on demand, you understand JOIN.

> **KR:** 모든 JOIN 종류는 사실 똑같은 하나의 질문에 대한 답입니다 — "매칭을 못 찾은 행은 어떻게 되어야 하는가?" — 그리고 `INNER`/`LEFT`/`RIGHT`/`FULL OUTER` 각각에 대해 이 질문에 즉시 답할 수 있다면, JOIN을 이해한 것입니다.

---
# ❓ Review Questions

**Q1.** For `orders` (5 rows, one with no matching customer) `JOIN`ed to `customers` (4 rows, two with no orders), how many rows does each of `INNER`, `LEFT` (orders-based), `RIGHT` (customers-based), and `FULL OUTER` return?  
**Q1.** `orders`(5행, 그중 하나는 매칭되는 고객 없음)와 `customers`(4행, 그중 둘은 주문 없음)를 `JOIN`할 때, `INNER`, `LEFT`(orders 기준), `RIGHT`(customers 기준), `FULL OUTER` 각각 몇 행이 나오는가?

INNER keeps only matching pairs (4 orders). LEFT keeps all 5 orders. RIGHT keeps all customers but multiplies by their orders (6). FULL OUTER keeps both leftovers too (7).  
매칭 4 + 왼쪽 남은 1 + 오른쪽 남은 2 = 바깥쪽 7입니다.

**Q2.** Why does `A RIGHT JOIN B` always produce the exact same result as `B LEFT JOIN A`?  
**Q2.** 왜 `A RIGHT JOIN B`는 항상 `B LEFT JOIN A`와 정확히 같은 결과를 내는가?

A RIGHT JOIN B keeps every row of B; B LEFT JOIN A does the same. They are mirrors.  
살리는 테이블이 B로 같으면 결과가 같습니다.

**Q3.** You joined `customers` → `orders` → `order_items`, and the row count came out higher than the number of orders. Is this a bug? Why or why not?  
**Q3.** `customers` → `orders` → `order_items`를 조인했더니 결과 행 수가 주문 건수보다 많이 나왔다. 이것은 버그인가? 왜 그런가, 혹은 왜 아닌가?

Not a bug: a 1:N join fans out rows (one order, many line items).  
주문보다 행이 많은 것은 품목이 여러 줄이라서입니다.

**Q4.** In a `SELF JOIN` on an `employees` table, why must you use `LEFT JOIN` instead of `INNER JOIN` to correctly include the one employee with no manager?  
**Q4.** `employees` 테이블의 `SELF JOIN`에서, 매니저가 없는 직원 한 명을 제대로 포함하려면 왜 `INNER JOIN`이 아니라 `LEFT JOIN`을 써야 하는가?

INNER JOIN drops the CEO because NULL never matches a manager id. LEFT JOIN keeps every employee and leaves manager columns NULL.  
짝이 없는 행을 지키려면 LEFT JOIN입니다.

**Q5.** You ran `SELECT o.order_id, c.name FROM orders o, customers c` and got 20 rows back instead of the ~5 you expected. What happened, and how would you fix the query?  
**Q5.** `SELECT o.order_id, c.name FROM orders o, customers c`를 실행했더니 예상한 5행 정도가 아니라 20행이 나왔다. 무슨 일이 일어난 것이며, 쿼리를 어떻게 고쳐야 하는가?

Comma-separated tables with no ON make a cartesian product (5 × 4 = 20). Add an explicit JOIN ... ON.  
에러는 안 나고 행만 곱해집니다. 반드시 ON을 쓰세요.

---
*📅 Try answering these again in a few days. / 며칠 후 다시 답해보세요.*